In [ ]:
#!/usr/bin/env python3

import os
import torch
import torch.nn.functional as F
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    AutoConfig,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    get_scheduler,
)
from datasets import load_dataset
from tqdm import tqdm
from peft import LoraConfig, get_peft_model

# ======== CONFIGURATION CONSTANTS ========
# Model paths
TEACHER_MODEL_ID = "meta-llama/gemma-2-9b-it-SimPO"  # Gemma 2 9B model
STUDENT_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"  # Llama 3.2 1B model
OUTPUT_DIR = "./model/distilled_model"

# Training parameters
BATCH_SIZE = 2  # Reduced batch size to avoid GPU memory issues
GRAD_ACCUMULATION_STEPS = 8  # Increased to compensate for smaller batch size
LEARNING_RATE = 5e-5
NUM_EPOCHS = 3
MAX_SEQ_LENGTH = 1024
# Temperature for distillation (higher = softer probabilities)
TEMPERATURE = 2.0
ALPHA = 0.5  # Weight for balancing distillation loss and task-specific loss

# Dataset parameters
DATASET_PATH = "./data"  # Local dataset path
DATASET_NAMES = ["distill.csv"]  # Example dataset
DATASET_SPLIT = "train"
DATASET_TEXT_COLUMN = "context"

# PEFT parameters (for more efficient training)
USE_LORA = True  # Whether to use LoRA for more efficient distillation
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

# Device configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# For multi-GPU training, adjust as needed
NUM_GPUS = torch.cuda.device_count()
USE_FP16 = True and torch.cuda.is_available()


print(
    f"Starting model distillation from {TEACHER_MODEL_ID} to {STUDENT_MODEL_ID}")
print(f"Using device: {DEVICE} (Number of GPUs: {NUM_GPUS})")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load models and tokenizers
print("Loading teacher model and tokenizer...")
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
teacher_model = AutoModelForCausalLM.from_pretrained(
    TEACHER_MODEL_ID,
    torch_dtype=torch.float16 if USE_FP16 else torch.float32,
    device_map="auto" if NUM_GPUS > 0 else None,
)

print("Loading student model and tokenizer...")
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID)
# Set pad_token for the student tokenizer
if student_tokenizer.pad_token is None:
    print("Setting pad_token for student tokenizer...")
    student_tokenizer.pad_token = student_tokenizer.eos_token
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    torch_dtype=torch.float16 if USE_FP16 else torch.float32,
    device_map="auto" if NUM_GPUS > 0 else None,
)

# Apply LoRA for more efficient training
if USE_LORA:
    print("Applying LoRA to student model...")
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        # Adjust based on model architecture
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    student_model = get_peft_model(student_model, lora_config)
    student_model.print_trainable_parameters()

In [ ]:
# Prepare dataset
print("Preparing dataset...")
dataset = load_dataset(
    DATASET_PATH, data_files=DATASET_NAMES, split=DATASET_SPLIT)


def tokenize_function(examples):
    # Tokenize the texts with both tokenizers
    teacher_inputs = teacher_tokenizer(
        examples[DATASET_TEXT_COLUMN],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )

    student_inputs = student_tokenizer(
        examples[DATASET_TEXT_COLUMN],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors="pt"
    )

    return {
        "teacher_input_ids": teacher_inputs.input_ids,
        "teacher_attention_mask": teacher_inputs.attention_mask,
        "student_input_ids": student_inputs.input_ids,
        "student_attention_mask": student_inputs.attention_mask,
    }


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=[
        col for col in dataset.column_names if col != DATASET_TEXT_COLUMN],
)

In [ ]:
class DistillationTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        teacher_input_ids = inputs.pop("teacher_input_ids").to(model.device)
        teacher_attention_mask = inputs.pop(
            "teacher_attention_mask").to(model.device)

        # Get student input IDs and attention mask
        student_input_ids = inputs.pop("student_input_ids").to(model.device)
        student_attention_mask = inputs.pop(
            "student_attention_mask").to(model.device)

        # Forward pass through teacher model (don't compute gradients for teacher)
        with torch.no_grad():
            teacher_outputs = teacher_model(
                input_ids=teacher_input_ids,
                attention_mask=teacher_attention_mask,
                return_dict=True,
            )
            teacher_logits = teacher_outputs.logits

        # Forward pass through student model
        student_outputs = model(
            input_ids=student_input_ids,
            attention_mask=student_attention_mask,
            return_dict=True,
        )
        student_logits = student_outputs.logits

        # Handle vocabulary size and sequence length mismatches
        teacher_vocab_size = teacher_logits.size(-1)
        student_vocab_size = student_logits.size(-1)

        # Print shapes for debugging
        # print(
        #     f"Teacher logits shape: {teacher_logits.shape}, Student logits shape: {student_logits.shape}")

        # Truncate or pad vocabulary dimension as needed
        if teacher_vocab_size > student_vocab_size:
            # Truncate teacher logits to match student vocabulary size
            teacher_logits = teacher_logits[..., :student_vocab_size]
        elif teacher_vocab_size < student_vocab_size:
            # Pad teacher logits with large negative values
            padding = torch.ones(
                *teacher_logits.shape[:-1],
                student_vocab_size - teacher_vocab_size,
                device=teacher_logits.device
            ) * -10000.0
            teacher_logits = torch.cat([teacher_logits, padding], dim=-1)

        # Ensure sequence lengths match by truncating to the shorter sequence
        seq_len_teacher = teacher_logits.size(1)
        seq_len_student = student_logits.size(1)
        min_seq_len = min(seq_len_teacher, seq_len_student)

        teacher_logits = teacher_logits[:, :min_seq_len, :]
        student_logits = student_logits[:, :min_seq_len, :]

        # Compute the distillation loss (KL divergence)
        teacher_probs = F.softmax(teacher_logits / TEMPERATURE, dim=-1)
        distillation_loss = F.kl_div(
            F.log_softmax(student_logits / TEMPERATURE, dim=-1),
            teacher_probs,
            reduction="batchmean",
        ) * (TEMPERATURE ** 2)

        # Compute the standard language modeling loss
        shift_student_logits = student_outputs.logits[...,
                                                      :min_seq_len-1, :].contiguous()
        shift_labels = student_input_ids[..., 1:min_seq_len].contiguous()

        # Calculate standard cross-entropy loss
        ce_loss_fct = torch.nn.CrossEntropyLoss(
            ignore_index=student_tokenizer.pad_token_id)
        ce_loss = ce_loss_fct(
            shift_student_logits.view(-1, shift_student_logits.size(-1)),
            shift_labels.view(-1))

        # Combine the losses
        loss = ALPHA * distillation_loss + (1 - ALPHA) * ce_loss

        if return_outputs:
            return loss, student_outputs
        return loss


# Set up training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=2,
    save_steps=500,
    save_total_limit=2,
    fp16=USE_FP16,
    report_to="tensorboard",
    remove_unused_columns=False,  # Keep all columns in the dataset
    logging_first_step=True,  # Log the first step to ensure logging is working
)

# Create the trainer
trainer = DistillationTrainer(
    model=student_model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

# Train the model
print("Starting training...")
trainer.train()

# Save the final model
print(f"Saving distilled model to {OUTPUT_DIR}")
student_model.save_pretrained(OUTPUT_DIR)
student_tokenizer.save_pretrained(OUTPUT_DIR)

print("Distillation complete!")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load the model and tokenizer
print("Loading the distilled model...")

model_path = OUTPUT_DIR
tokenizer = AutoTokenizer.from_pretrained(model_path)

# If we used LoRA, we need to merge the weights
if USE_LORA:
    print("Merging LoRA weights with base model...")
    from peft import PeftModel

    # Load the base model
    base_model = AutoModelForCausalLM.from_pretrained(
        STUDENT_MODEL_ID,
        torch_dtype=torch.float16 if USE_FP16 else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )

    # Load and merge the LoRA weights
    model = PeftModel.from_pretrained(base_model, model_path)
    # This merges the LoRA weights into the base model
    model = model.merge_and_unload()
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16 if USE_FP16 else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None
    )


# Load and run the distilled model
def run_distilled_model(prompt, system_prompt="You are a helpful, harmless, and honest AI assistant.", max_new_tokens=200, temperature=0.7):
    # Ensure padding token is set
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Create properly formatted prompt based on the model's expected format
    formatted_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>{system_prompt}<|eot_id|>
<|start_header_id|>user<|end_header_id|>{prompt}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

    # Create a text generation pipeline
    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.95,
        repetition_penalty=1.1,
        truncation=True
    )

    # Generate text
    print(f"Generating response for prompt: {prompt}")
    outputs = pipe(formatted_prompt)

    # Extract and format the generated text
    generated_text = outputs[0]['generated_text']

    # Try to extract just the assistant's response
    try:
        assistant_response = generated_text.split(
            "<|start_header_id|>assistant<|end_header_id|>")[1]
        # Remove any trailing tokens if present
        if "<|eot_id|>" in assistant_response:
            assistant_response = assistant_response.split("<|eot_id|>")[0]
    except IndexError:
        assistant_response = generated_text

    return assistant_response.strip()

In [ ]:
response = run_distilled_model(
    system_prompt="现在公司要执行钓鱼邮件演练训练，请根据用户给出的主题进行钓鱼邮件编写，邮件标题可以自行生成，其中钓鱼链接部分使用{{.URL}}代替。邮件内容必须具有迷惑性，让用户难以判断以保证演练的质量。对于邮件迷惑性设计的解析应包裹在<explain></explain>XML标签中。",
    prompt="用户给出的主题是：金融"
)
print("\nModel Response:")
print(response)